# Credit Modeling

## Packages

In [382]:


#importing Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from sklearn.metrics import roc_auc_score,classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from warnings import filterwarnings
filterwarnings('ignore')


pd.set_option('display.max_columns', None)

print("Done importing")

Done importing


## Data Cleaning

### Data Loading

In [ ]:
# Loading loan application data and having a view of data
loan_book = pd.read_csv("../data/loan_book.csv", index_col=0)
loan_book.head(5)

,age,annual_income,employment_length_years,home_ownership,region,num_open_accounts,num_delinquencies_2yr,total_revolving_balance,credit_utilisation_pct,months_since_oldest_account,num_hard_inquiries_6mo,loan_amount,interest_rate,loan_purpose,dti_ratio,months_since_last_delinquency,pct_accounts_current,application_date,application_dow,branch_code_id,months_at_current_address,email_domain_type,phone_verified,default_flag,set
applicant_id_hash,,,,,,,,,,,,,,,,,,,,,,,,,
11a2f242b28a331c,35.5,29401.0,3.6,MORTGAGE,North-Urban,9.0,7,1778.0,12.7,177.0,2,16326.0,21.11,major_purchase,0.245,1.0,50.9,06/14/2021,Monday,347,119,other,True,0,test
5afff059dc04c6f0,25.0,32005.0,1.4,mortgage,South-Urban,10.0,0,0.0,31.2,42.0,1,8293.0,12.69,medical,0.337,105.0,93.0,2021-08-28,Friday,367,7,free,False,0,train
eb85b183be0505f3,40.0,26730.0,NaN,MORTGAGE,East-Urban,9.0,0,3690.0,29.4,148.0,0,13080.0,10.45,debt_consolidation,0.124,NaN,87.9,2021-02-23,Saturday,895,142,free,True,0,test
7d99629563bdb528,30.9,60105.0,7.5,MORTGAGE,South-Suburban,7.0,0,7636.0,48.8,115.0,2,6752.0,15.26,other,0.218,58.0,96.1,2021-10-07,Thursday,387,69,other,True,1,test
658f1f05e484ce14,49.1,112275.0,6.9,MORTGAGE,East-Suburban,10.0,0,14450.0,68.2,324.0,3,19144.0,9.04,other,0.195,NaN,90.8,23-Aug-2021,Sunday,561,182,free,True,0,test


Now we check the shape of the data set

In [384]:
loan_book.shape

(120960, 25)

The data set contains **25 Variables** and **120358 observations(applicants)**

### Duplicates

Checking for number of duplicates in data set.

In [385]:
# number of duplicates in data set
loan_book.duplicated().sum()

np.int64(602)

The data set (loan book) contains **602 duplicates**, so we are going to drop them.

In [386]:
# Dropping duplicates
loan_book.drop_duplicates(inplace=True)
loan_book.reset_index(inplace=True)
loan_book.set_index("applicant_id_hash", inplace=True)
loan_book.duplicated().sum()

np.int64(0)

### Missing Values

Checking how many variables(Features) have missing values

In [387]:
# variables with missing values
missing = loan_book.isnull().sum()
missing[missing>0]

annual_income                     8638
employment_length_years           3711
num_open_accounts                 2423
months_since_last_delinquency    60068
dtype: int64

 There are **4 variables** with missing values.

#### Imputing annual_income

So i will fill the missing values by **grouping by region** and fill a  with **median**. Salary can vary with region

In [388]:
# filling using by grouping by region and use median of region
loan_book["annual_income"] = loan_book.groupby("region")["annual_income"].transform( lambda x : x.fillna(x.median()) )

#### imputing employment_length_years and num_open_accounts

Filling the missing values using **median** for employment_length_years, num_open_accounts and months_since_last_delinquency

In [389]:
# filling missing values using median
loan_book["employment_length_years"] = loan_book["employment_length_years"].fillna(
    loan_book["employment_length_years"].median()
)

loan_book["num_open_accounts"] = loan_book["num_open_accounts"].fillna(
    loan_book["num_open_accounts"].median()
)


# dropping months_since_last_delinquency

In [390]:
loan_book.drop(columns="months_since_last_delinquency", inplace=True)

### Data types

Checking if variables have correct data type

In [391]:
# data type
loan_book.dtypes

age                            float64
annual_income                  float64
employment_length_years        float64
home_ownership                  object
region                          object
num_open_accounts              float64
num_delinquencies_2yr            int64
total_revolving_balance        float64
credit_utilisation_pct         float64
months_since_oldest_account    float64
num_hard_inquiries_6mo           int64
loan_amount                    float64
interest_rate                  float64
loan_purpose                    object
dti_ratio                      float64
pct_accounts_current           float64
application_date                object
application_dow                 object
branch_code_id                   int64
months_at_current_address        int64
email_domain_type               object
phone_verified                    bool
default_flag                     int64
set                             object
dtype: object

  Variables **age**, **employment_length_years**, **num_open_accounts**, **months_since_oldest_account**
   have float data type but instead should be integer and **application_date** has str(object) data type, instead it should be date data
   type.

In [392]:
# fixing data types
# float to int
v = [ "age","employment_length_years", "num_open_accounts", "months_since_oldest_account"]
for var in v:
    loan_book[var] = loan_book[var].astype("int64")

# str(object) to date data type
loan_book["application_date"]=loan_book["application_date"].astype("datetime64[ns]")

### Invalid and inconsistency values

We checking for inconsistency and invalid values in our categorical variables

In [393]:
#  selecting categorical variables variables
cat_var = loan_book.select_dtypes(exclude=["int64","float64","bool","datetime64[ns]"]).columns

# checking unique values
for var in cat_var:
    print(f"\033[1m{var}\033[0m: {np.array(loan_book[var].unique())}")

home_ownership: ['MORTGAGE' 'mortgage' 'RENT' 'OWN' 'OTHER' 'rent' 'Owner' 'Renting' 'own'
 'Own' 'Rent' 'Mortgage' 'other' 'Other']
region: ['North-Urban' 'South-Urban' 'East-Urban' 'South-Suburban' 'East-Suburban'
 'West-Urban' 'Central-Urban' 'North-Suburban' 'West-Suburban'
 'Central-Suburban']
loan_purpose: ['major_purchase' 'medical' 'debt_consolidation' 'other'
 'home_improvement' 'DEBT_CONSOLIDATION' 'education' 'small_business'
 'Home Improvement' 'Debt Consolidation' 'home improvement' 'OTHER'
 'Major Purchase' 'major purchase' 'debt consolidation' 'Small Business'
 'MEDICAL' 'Education' 'Other' 'Medical' 'small business']
application_dow: ['Monday' 'Friday' 'Saturday' 'Thursday' 'Sunday' 'Wednesday' 'Tuesday']
email_domain_type: ['other' 'free' 'corporate']
set: ['test' 'train']


We can see that **home_ownership**, **loan_purpose** have inconsistency formats of labels. labels must have same labels

In [394]:
# fixing labels in home_ownership
loan_book["home_ownership"] = loan_book["home_ownership"].str.capitalize().map(lambda x: x if x not in ["Rent","Own"]
else  ("Renting" if x=="Rent" else "Owner") )

# fixing loan_purpose
loan_book["loan_purpose"] = loan_book["loan_purpose"].str.capitalize().map(lambda x: x.replace("_"," "))

Checking if things still looks okay

In [395]:
loan_book

,age,annual_income,employment_length_years,home_ownership,region,num_open_accounts,num_delinquencies_2yr,total_revolving_balance,credit_utilisation_pct,months_since_oldest_account,num_hard_inquiries_6mo,loan_amount,interest_rate,loan_purpose,dti_ratio,pct_accounts_current,application_date,application_dow,branch_code_id,months_at_current_address,email_domain_type,phone_verified,default_flag,set
applicant_id_hash,,,,,,,,,,,,,,,,,,,,,,,,
11a2f242b28a331c,35,29401.0,3,Mortgage,North-Urban,9,7,1778.0,12.7,177,2,16326.0,21.11,Major purchase,0.245,50.9,2021-06-14,Monday,347,119,other,True,0,test
5afff059dc04c6f0,25,32005.0,1,Mortgage,South-Urban,10,0,0.0,31.2,42,1,8293.0,12.69,Medical,0.337,93.0,2021-08-28,Friday,367,7,free,False,0,train
eb85b183be0505f3,40,26730.0,5,Mortgage,East-Urban,9,0,3690.0,29.4,148,0,13080.0,10.45,Debt consolidation,0.124,87.9,2021-02-23,Saturday,895,142,free,True,0,test
7d99629563bdb528,30,60105.0,7,Mortgage,South-Suburban,7,0,7636.0,48.8,115,2,6752.0,15.26,Other,0.218,96.1,2021-10-07,Thursday,387,69,other,True,1,test
658f1f05e484ce14,49,112275.0,6,Mortgage,East-Suburban,10,0,14450.0,68.2,324,3,19144.0,9.04,Other,0.195,90.8,2021-08-23,Sunday,561,182,free,True,0,test
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
f72c2a5472b39f2a,24,80416.0,3,Owner,South-Urban,11,2,5352.0,64.9,48,2,9182.0,12.23,Other,0.165,73.7,2021-09-19,Tuesday,400,9,corporate,True,0,train
88b6b962fbd4c70d,21,52346.0,1,Mortgage,West-Suburban,12,0,1218.0,51.3,63,1,10338.0,11.43,Home improvement,0.221,53.9,2022-04-11,Monday,802,216,corporate,True,0,train
8c9f6907e46e6149,53,45310.0,7,Owner,West-Suburban,13,3,1757.0,22.7,284,2,6706.0,19.75,Debt consolidation,0.270,52.5,2021-12-03,Sunday,777,51,free,False,0,train


## EDA

EDA was done in the smart view app that was built specifically for this analysis: [Smart View](https://smart-view.streamlit.app/)

## Custom functions

### function for calculating Weight of Evidence and Infomation Value

In [396]:
def woe_iv(data, feature, target):
    """for WoE and Iv calculations"""
    df = data[[feature, target]].dropna().copy()

    grouped = df.groupby(feature,observed=False)[target].agg(['count', 'sum'])
    grouped.columns = ['total', 'bad']

    grouped['good'] = grouped['total'] - grouped['bad']

    grouped['dist_good'] = grouped['good'] / grouped['good'].sum()
    grouped['dist_bad'] = grouped['bad'] / grouped['bad'].sum()

    grouped['WoE'] = np.log(
        (grouped['dist_good'] + 1e-6) /
        (grouped['dist_bad'] + 1e-6)
    )

    # IV contribution per bin
    grouped['iv_bin'] = (
        grouped['dist_good'] - grouped['dist_bad']
    ) * grouped['WoE']

    return grouped.reset_index()

### function for binning a numeric value

In [397]:
def bin_numeric(bin_edges: pd.Series | list | np.ndarray,
                data: pd.DataFrame,
                feature: str,target: str):

    bin_edges = np.sort(np.array(bin_edges))

    bins = [data[feature].min()] + list(bin_edges) + [data[feature].max()]
    new_df = pd.DataFrame()

    new_df[feature] = pd.cut(
        data[feature],
        bins=bins,
        include_lowest=True,
        duplicates="drop"
    )
    new_df[target] = data[target]

    return new_df

### function for calculating the midpoint in the bin

In [398]:
def midpoint_bin(data: pd.DataFrame,feature: str):
    new_df = data.copy()
    # midpoint calculation
    new_df["bin_midpoint"] = new_df[feature].apply(
        lambda x: (x.left + x.right) / 2 if pd.notnull(x) else np.nan
    )

    return new_df

### function for capping variable

In [399]:
def capping_var(df_train:pd.DataFrame,df_test:pd.DataFrame,var:str):
    """
    Used to cap  variable with outliers
    :param df_train: Dataframe of the data with the variable of interest (train set)
    :param df_test: Dataframe of the data with the variable of interest (test set)
    :param var: name of the variable of interest
    :return: new dataframe with the variable of interest capped
    """
    df_train,df_test = df_train.copy(),df_test.copy()
    Q1 = df_train[var].quantile(0.25)
    Q3 = df_train[var].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df_train[f"{var}_capped"] = df_train[var].clip(lower, upper)
    df_test[f"{var}_capped"] = df_test[var].clip(lower, upper)
    return df_train,df_test

### function for to evaluating predictions

In [400]:
def auc_evaluator(model,X_test,y_test):
    y_prob = model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test,y_prob)
    return score

### function for replacing orginal values with WoE values

In [401]:
def apply_woe(df, original_var, bin_to_woe_dict, bins=None):
    """
    Replace original variable with its WoE value
    """

    df = df.copy()
    bins = [df[original_var].min()] + list(bins) + [df[original_var].max()]
    # Create binned version and map WoE
    df[f'{original_var}_woe'] = pd.cut(df[original_var],
                                       bins=bins,
                                       labels=list(bin_to_woe_dict.keys()),
                                       right=True,include_lowest=True,duplicates="drop").map(bin_to_woe_dict)
    df.drop(columns = original_var, inplace = True)

    return df

## Model Build and Feature Engineering

### Train and Test Dataset

In [402]:
train = loan_book.loc[loan_book["set"] == "train"]
test = loan_book.loc[loan_book["set"] == "test"]

First I will remove some of the variable because they are forbidden features or proxy ,and some they lack business interpretability for credit modelling. The following variable **region**,**email_domain_type**,**branch_code_id**,**application_date**,**application_dow**

In [403]:
rev = ["branch_code_id","application_date","application_dow","loan_purpose","region","email_domain_type","interest_rate","set","home_ownership",
       "phone_verified","months_at_current_address","months_since_oldest_account"]
train.drop(columns = rev, inplace = True)
test.drop(columns = rev, inplace = True)

### Features and Target

In [404]:
y_train = train["default_flag"]
y_test = test["default_flag"]
X_train = train.drop(columns = ["default_flag"])
X_test = test.drop(columns = ["default_flag"])


In [405]:
scaler = StandardScaler()
X_scaled_train_1 = scaler.fit_transform(X_train)
X_scaled_test_1 = scaler.transform(X_test)

### Baseline model

In [406]:
model_base = LogisticRegression(max_iter=10000,class_weight="balanced",random_state=42,verbose=1,solver="sag")
model_base.fit(X_scaled_train_1,y_train)

convergence after 36 epochs took 1 seconds


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [407]:
base_score = auc_evaluator(model_base,X_scaled_test_1,y_test)
np.round(base_score,2)

np.float64(0.76)

In [408]:
print(classification_report(y_test, model_base.predict(X_scaled_test_1)))

              precision    recall  f1-score   support

           0       0.93      0.70      0.80     30523
           1       0.30      0.69      0.41      5561

    accuracy                           0.70     36084
   macro avg       0.61      0.70      0.61     36084
weighted avg       0.83      0.70      0.74     36084



### Model 1 and new Feature

#### Feature Engineering

creating age_WoE and  replace the orginal age variable

In [409]:
eges = [25,31,43]    # split points
age_bins = bin_numeric(eges,train,"age","default_flag")      # making bins
age_bins["age"].value_counts()

age
(31.0, 43.0]      32360
(43.0, 70.0]      26089
(20.999, 25.0]    13278
(25.0, 31.0]      12547
Name: count, dtype: int64

calculating the Weight of evidence of the created bins

In [410]:
woe_result = midpoint_bin(woe_iv(age_bins,"age","default_flag"),"age")      # WoE calculation
woe_result

,age,total,bad,good,dist_good,dist_bad,WoE,iv_bin,bin_midpoint
0,"(20.999, 25.0]",13278,3825,9453,0.132663,0.293824,-0.795168,0.128150,22.9995
1,"(25.0, 31.0]",12547,2487,10060,0.141181,0.191043,-0.302454,0.015081,28.0000
2,"(31.0, 43.0]",32360,4306,28054,0.393707,0.330773,0.174175,0.010962,37.0000
3,"(43.0, 70.0]",26089,2400,23689,0.332449,0.184360,0.589594,0.087312,56.5000


replacing the orginal values of age with  age bins WoE

In [411]:
# Create dictionary for encoding
bin_to_woe = dict(zip(woe_result["age"], woe_result['WoE']))   # creating a map used in mapping
X_train_2 = apply_woe(X_train,"age",bin_to_woe,eges)  # new value are introduce instead of the existing ones
X_test_2 = apply_woe(X_test,"age",bin_to_woe,eges)
X_scaled_train = scaler.fit_transform(X_train_2)      #scaling
X_scaled_test = scaler.transform(X_test_2)

#### fitting the  model 1

In [412]:
model_1 = LogisticRegression(max_iter=10000,class_weight="balanced",random_state=42,verbose=1,solver="sag")
model_1.fit(X_scaled_train,y_train)          #fitting the model

convergence after 51 epochs took 1 seconds


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [413]:
score_1 = auc_evaluator(model_1,X_scaled_test,y_test)
np.round(score_1,2)

np.float64(0.76)

In [414]:
print(classification_report(y_test, model_1.predict(X_scaled_test)))

              precision    recall  f1-score   support

           0       0.92      0.70      0.80     30523
           1       0.29      0.68      0.41      5561

    accuracy                           0.70     36084
   macro avg       0.61      0.69      0.60     36084
weighted avg       0.83      0.70      0.74     36084



### Model 2 and new feature

#### feature engineering

Capping  and Logging annual income

In [415]:
# capping and loging annual income

X_train_3,X_test_3 = capping_var(X_train_2,X_test_2,"annual_income")     # capping
X_train_3["log_annual_income_capped"] = X_train_3["annual_income_capped"].transform(lambda x: np.log(x))  #loging
X_train_3.drop(columns=["annual_income","annual_income_capped"],inplace=True)

X_test_3["log_annual_income_capped"] = X_test_3["annual_income_capped"].transform(lambda x: np.log(x))
X_test_3.drop(columns=["annual_income","annual_income_capped"],inplace=True)

In [416]:
X_scaled_train = scaler.fit_transform(X_train_3)
X_scaled_test = scaler.transform(X_test_3)

#### fitting the model 2

In [417]:
model_2 = LogisticRegression(max_iter=10000,class_weight="balanced",random_state=42,verbose=1,solver="sag")
model_2.fit(X_scaled_train,y_train)

convergence after 37 epochs took 0 seconds


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [418]:
score_2 = auc_evaluator(model_2,X_scaled_test,y_test)
np.round(score_2,2)

np.float64(0.78)

In [419]:
print(classification_report(y_test, model_2.predict(X_scaled_test)))

              precision    recall  f1-score   support

           0       0.93      0.72      0.81     30523
           1       0.31      0.70      0.43      5561

    accuracy                           0.71     36084
   macro avg       0.62      0.71      0.62     36084
weighted avg       0.83      0.71      0.75     36084



### Model 3 with new feature

#### feature engineering

removing a loging total revolving balance and replacing it with log of total revolving balance also loan amount

In [420]:
X_train_4,X_test_4 = X_train_3.copy(),X_test_3.copy(),
# removing a loging total revolving balance and replacing it with log of total revolving balance also loan amount
X_train_4["log_total_revolving_balance"] = X_train_4["total_revolving_balance"].transform(lambda x: np.log(x+1))
X_train_4["log_loan_amount"] = X_train_4["loan_amount"].transform(lambda x: np.log(x))
X_test_4["log_total_revolving_balance"] = X_test_4["total_revolving_balance"].transform(lambda x: np.log(x+1))
X_test_4["log_loan_amount"] = X_test_4["loan_amount"].transform(lambda x: np.log(x))

X_train_4.drop(columns=["total_revolving_balance","loan_amount"],inplace=True)
X_test_4.drop(columns=["total_revolving_balance","loan_amount"],inplace=True)

In [421]:
X_scaled_train = scaler.fit_transform(X_train_4)
X_scaled_test = scaler.transform(X_test_4)

#### fitting the  model 3

In [422]:
model_3 = LogisticRegression(max_iter=10000,class_weight="balanced",random_state=42,verbose=1,solver="sag")
model_3.fit(X_scaled_train,y_train)

convergence after 33 epochs took 0 seconds


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [423]:
score_3 = auc_evaluator(model_3,X_scaled_test,y_test)
np.round(score_3,2)

np.float64(0.78)

In [424]:
print(classification_report(y_test, model_3.predict(X_scaled_test)))

              precision    recall  f1-score   support

           0       0.93      0.72      0.81     30523
           1       0.31      0.70      0.43      5561

    accuracy                           0.71     36084
   macro avg       0.62      0.71      0.62     36084
weighted avg       0.83      0.71      0.75     36084



### Model 4 with new feature

#### feature engineering

replaceing the num_open_accounts by WoE values

In [425]:
X_train_5,X_test_5 = X_train_4.copy(),X_test_4.copy()
accts_eges = np.histogram_bin_edges(train["num_open_accounts"],bins=5)[:-2]      # tried finding the optimal using the EDA app
accts_bins = bin_numeric(accts_eges,train,"num_open_accounts","default_flag")
accts_bins["num_open_accounts"].value_counts()

num_open_accounts
(4.6, 9.2]       52668
(9.2, 13.8]      20574
(-0.001, 4.6]     8156
(13.8, 23.0]      2876
Name: count, dtype: int64

In [426]:
woe_result_accts = woe_iv(accts_bins,"num_open_accounts","default_flag")
bin_to_woe_accts = dict(zip(woe_result_accts["num_open_accounts"], woe_result_accts['WoE']))
bin_to_woe_accts

{Interval(-0.001, 4.6, closed='right'): -0.3226666532293364,
 Interval(4.6, 9.2, closed='right'): 0.18446730149089013,
 Interval(9.2, 13.8, closed='right'): -0.084546666619138,
 Interval(13.8, 23.0, closed='right'): -1.0967388900673651}

In [427]:
X_train_5= apply_woe(X_train_5,"num_open_accounts",bin_to_woe_accts,accts_eges)
X_test_5=apply_woe(X_test_5,"num_open_accounts",bin_to_woe_accts,accts_eges)

In [428]:
X_scaled_train = scaler.fit_transform(X_train_5)
X_scaled_test = scaler.transform(X_test_5)

#### fitting model 4

In [429]:
model_4 = LogisticRegression(max_iter=10000,class_weight="balanced",random_state=42,verbose=1,solver="sag")
model_4.fit(X_scaled_train,y_train)

convergence after 35 epochs took 1 seconds


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [430]:
score_4 = auc_evaluator(model_4,X_scaled_test,y_test)
np.round(score_4,2)

np.float64(0.79)

In [431]:
print(classification_report(y_test, model_4.predict(X_scaled_test)))

              precision    recall  f1-score   support

           0       0.93      0.73      0.82     30523
           1       0.32      0.71      0.44      5561

    accuracy                           0.72     36084
   macro avg       0.63      0.72      0.63     36084
weighted avg       0.84      0.72      0.76     36084



### Model 5 and new feature

#### feature engineering

replacing the crediti_utilisation_pct with WoE values

In [432]:
X_train_6,X_test_6 = X_train_5.copy(),X_test_5.copy()
crd_util_eges = np.histogram_bin_edges(train["credit_utilisation_pct"],bins=8)[:-2]
crd_util_bins = bin_numeric(crd_util_eges,train,"credit_utilisation_pct","default_flag")
crd_util_bins["credit_utilisation_pct"].value_counts()

credit_utilisation_pct
(26.55, 39.825]     20491
(39.825, 53.1]      19604
(13.275, 26.55]     14813
(53.1, 66.375]      14165
(66.375, 79.65]      7412
(-0.001, 13.275]     5424
(79.65, 106.2]       2365
Name: count, dtype: int64

In [433]:
woe_result_crd_util = woe_iv(crd_util_bins,"credit_utilisation_pct","default_flag")
bin_to_woe_crd_util = dict(zip(woe_result_crd_util["credit_utilisation_pct"], woe_result_crd_util['WoE']))
bin_to_woe_crd_util

{Interval(-0.001, 13.275, closed='right'): 0.17217245002674864,
 Interval(13.275, 26.55, closed='right'): 0.174818302222657,
 Interval(26.55, 39.825, closed='right'): 0.1397720093561652,
 Interval(39.825, 53.1, closed='right'): 0.060737244548101726,
 Interval(53.1, 66.375, closed='right'): -0.07115251739403798,
 Interval(66.375, 79.65, closed='right'): -0.38351849613073374,
 Interval(79.65, 106.2, closed='right'): -0.9329022186467583}

In [434]:
X_train_6= apply_woe(X_train_6,"credit_utilisation_pct",bin_to_woe_crd_util,crd_util_eges)
X_test_6=apply_woe(X_test_6,"credit_utilisation_pct",bin_to_woe_crd_util,crd_util_eges)

In [435]:
X_scaled_train = scaler.fit_transform(X_train_6)
X_scaled_test = scaler.transform(X_test_6)

#### fitting a model 5

In [436]:
model_5 = LogisticRegression(max_iter=10000,class_weight="balanced",random_state=42,verbose=1,solver="sag")
model_5.fit(X_scaled_train,y_train)

convergence after 34 epochs took 1 seconds


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [437]:
score_5= auc_evaluator(model_5,X_scaled_test,y_test)
np.round(score_5,2)

np.float64(0.79)

In [438]:
print(classification_report(y_test, model_5.predict(X_scaled_test)))

              precision    recall  f1-score   support

           0       0.93      0.73      0.82     30523
           1       0.32      0.71      0.44      5561

    accuracy                           0.73     36084
   macro avg       0.63      0.72      0.63     36084
weighted avg       0.84      0.73      0.76     36084



### Model 6 with new feature

#### feature engineering

replacing the employment_length_years with WoE values

In [439]:
X_train_7,X_test_7 = X_train_6.copy(),X_test_6.copy()
employ_eges = [2,5,8,12]
employ_bins = bin_numeric(employ_eges,train,"employment_length_years","default_flag")
employ_bins["employment_length_years"].value_counts()

employment_length_years
(2.0, 5.0]       28734
(5.0, 8.0]       19771
(-0.001, 2.0]    19395
(8.0, 12.0]      11327
(12.0, 30.0]      5047
Name: count, dtype: int64

In [440]:
woe_result_employ = woe_iv(employ_bins,"employment_length_years","default_flag")
bin_to_woe_employ = dict(zip(woe_result_employ["employment_length_years"], woe_result_employ['WoE']))
bin_to_woe_employ

{Interval(-0.001, 2.0, closed='right'): -0.5756616592552324,
 Interval(2.0, 5.0, closed='right'): 0.07819657538836877,
 Interval(5.0, 8.0, closed='right'): 0.2732354077430899,
 Interval(8.0, 12.0, closed='right'): 0.44887882197652906,
 Interval(12.0, 30.0, closed='right'): 0.4791283148216141}

In [441]:
X_train_7= apply_woe(X_train_7,"employment_length_years",bin_to_woe_employ,employ_eges)
X_test_7=apply_woe(X_test_7,"employment_length_years",bin_to_woe_employ,employ_eges)

In [442]:
X_scaled_train = scaler.fit_transform(X_train_7)
X_scaled_test = scaler.transform(X_test_7)

#### fitting model 6

In [443]:
model_6 = LogisticRegression(max_iter=10000,class_weight="balanced",random_state=42,verbose=1,solver="sag")
model_6.fit(X_scaled_train,y_train)

convergence after 33 epochs took 1 seconds


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [444]:
score_6= auc_evaluator(model_6,X_scaled_test,y_test)
np.round(score_6,2)

np.float64(0.79)

In [445]:
print(classification_report(y_test, model_6.predict(X_scaled_test)))

              precision    recall  f1-score   support

           0       0.93      0.73      0.82     30523
           1       0.32      0.71      0.44      5561

    accuracy                           0.73     36084
   macro avg       0.63      0.72      0.63     36084
weighted avg       0.84      0.73      0.76     36084



### Final Model

In [446]:
X_train_final ,X_test_final =X_train_7.copy(),X_test_7.copy()
X_scaled_train_final = scaler.fit_transform(X_train_final)
X_scaled_test_final = scaler.transform(X_test_final)

In [447]:
final_model = LogisticRegression(max_iter=10000,class_weight="balanced",random_state=42,verbose=1,solver="sag")
final_model.fit(X_scaled_train_final,y_train)

convergence after 33 epochs took 0 seconds


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [448]:
score_final= auc_evaluator(final_model,X_scaled_test_final,y_test)
np.round(score_final,2)

np.float64(0.79)

In [449]:
score_final

0.7904035187605718

In [450]:
print(classification_report(y_test, final_model.predict(X_scaled_test_final)))

              precision    recall  f1-score   support

           0       0.93      0.73      0.82     30523
           1       0.32      0.71      0.44      5561

    accuracy                           0.73     36084
   macro avg       0.63      0.72      0.63     36084
weighted avg       0.84      0.73      0.76     36084



## Coefficients

In [451]:
coefficients = final_model.coef_[0]
intercept = final_model.intercept_[0]

In [452]:

feature_names = X_train_final.columns

coefficients = final_model.coef_[0]
intercept = final_model.intercept_[0]

# Create table
coef_table = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients.round(4),
    'Odds Ratio': np.exp(coefficients).round(3),
    '|Coefficient|': abs(coefficients).round(4)
}).sort_values(by='|Coefficient|', ascending=False)

print("Intercept:", round(intercept, 4))
print(coef_table)

Intercept: -0.4299
                        Feature  Coefficient  Odds Ratio  |Coefficient|
5      log_annual_income_capped      -0.5912       0.554         0.5912
0         num_delinquencies_2yr       0.5796       1.785         0.5796
4                       age_woe      -0.3894       0.677         0.3894
8         num_open_accounts_woe      -0.3274       0.721         0.3274
1        num_hard_inquiries_6mo       0.2857       1.331         0.2857
9    credit_utilisation_pct_woe      -0.2792       0.756         0.2792
10  employment_length_years_woe      -0.2373       0.789         0.2373
2                     dti_ratio       0.1308       1.140         0.1308
3          pct_accounts_current      -0.0910       0.913         0.0910
6   log_total_revolving_balance      -0.0246       0.976         0.0246
7               log_loan_amount       0.0179       1.018         0.0179
